# Federated Bloom LoRA — GPU Training (Colab)

**Goal:** Train the privacy-preserving federated adapter on GPU, evaluate on the held-out test split, and download the adapter for local CPU deployment.

**Before running:** Runtime → Change runtime type → **T4 GPU**

**Expected runtime:** ~15–45 min (4 clients × 3 rounds) on T4 vs many hours on CPU.

In [ ]:
# Option A: clone your repo (replace URL)
# !git clone https://github.com/YOUR_USER/Framework.git
# %cd Framework

# Option B: upload Framework.zip to Colab files, then:
# !unzip -q Framework.zip
# %cd Framework

import os
assert os.path.isdir("federated"), "Run this notebook from the Framework repo root (or %cd Framework first)"

In [ ]:
!pip install -q transformers==4.41.0 peft==0.11.1 accelerate==0.33.0 \
    pandas scikit-learn matplotlib safetensors huggingface-hub tokenizers

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: enable GPU runtime (Runtime → Change runtime type → T4 GPU)")

In [ ]:
# Smoke test (2 clients, 1 round) — uncomment to verify loss drops below ~1.0 first
# !python federated/run_simulation.py --clients 2 --rounds 1 --eval-each-round

# Full federated run + merge + test eval + zip adapter
!python federated/run_gpu_pipeline.py --clients 4 --rounds 3 --eval-each-round

In [ ]:
import json
from pathlib import Path

sim = json.loads(Path("results/federated_lora_simulation.json").read_text())
print("Convergence curve:")
for row in sim.get("history", []):
    print(row)

metrics_path = Path("results/federated_bloom_test/metrics.json")
if metrics_path.is_file():
    print("\nTest metrics:", json.loads(metrics_path.read_text()))

In [ ]:
from google.colab import files

files.download("results/qwen_bloom_federated_adapter.zip")
files.download("results/federated_lora_simulation.json")
if Path("results/federated_bloom_test/metrics.json").is_file():
    files.download("results/federated_bloom_test/metrics.json")